# Text Classification with TF-IDF

In this notebook we classify newsgroup posts into 4 categories using TF-IDF features and compare three classifiers:
- **Naive Bayes** (MultinomialNB)
- **Logistic Regression**
- **Support Vector Machine** (LinearSVC)

Dataset: sklearn's 20 Newsgroups (4 selected categories).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)
from sklearn.model_selection import cross_val_score

sns.set_style('whitegrid')
print('Imports ready.')

## 1. Load Data

In [ ]:
categories = ['sci.space', 'rec.sport.baseball', 'comp.graphics', 'talk.politics.guns']

train_data = fetch_20newsgroups(subset='train', categories=categories,
                                 remove=('headers', 'footers', 'quotes'))
test_data = fetch_20newsgroups(subset='test', categories=categories,
                                remove=('headers', 'footers', 'quotes'))

print(f'Training samples: {len(train_data.data)}')
print(f'Test samples:     {len(test_data.data)}')
print(f'Categories:       {train_data.target_names}')

## 2. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, stop_words='english',
                         max_df=0.95, min_df=2)

X_train = tfidf.fit_transform(train_data.data)
X_test = tfidf.transform(test_data.data)
y_train = train_data.target
y_test = test_data.target

print(f'TF-IDF matrix shape (train): {X_train.shape}')
print(f'TF-IDF matrix shape (test):  {X_test.shape}')

## 3. Train and Evaluate Classifiers

In [ ]:
classifiers = {
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0),
    'SVM (LinearSVC)': LinearSVC(max_iter=2000, C=1.0),
}

results = {}
predictions = {}

for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    predictions[name] = y_pred
    print(f'{name:25s}  Accuracy: {acc:.4f}')

## 4. Accuracy Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
names = list(results.keys())
accs = list(results.values())
colors = ['#3498db', '#2ecc71', '#e74c3c']
bars = ax.bar(names, accs, color=colors, edgecolor='black', linewidth=0.5)

for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{acc:.3f}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.set_title('Classifier Accuracy Comparison')
plt.tight_layout()
plt.show()

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=[c.split('.')[-1] for c in categories],
                yticklabels=[c.split('.')[-1] for c in categories])
    axes[idx].set_title(f'{name}\n(Acc: {results[name]:.3f})')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')

plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Detailed Classification Reports

In [ ]:
# Pick the best classifier for the detailed report
best_name = max(results, key=results.get)
print(f'Best classifier: {best_name} ({results[best_name]:.4f})\n')
print(classification_report(y_test, predictions[best_name],
                            target_names=categories))

## 7. Per-Class Accuracy Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(categories))
width = 0.25

for i, (name, y_pred) in enumerate(predictions.items()):
    per_class_acc = []
    for cls_idx in range(len(categories)):
        mask = y_test == cls_idx
        acc = (y_pred[mask] == y_test[mask]).mean()
        per_class_acc.append(acc)
    ax.bar(x + i * width, per_class_acc, width, label=name, color=colors[i])

ax.set_xticks(x + width)
ax.set_xticklabels([c.split('.')[-1] for c in categories], rotation=15)
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1.1)
ax.set_title('Per-Class Accuracy by Classifier')
ax.legend()
plt.tight_layout()
plt.show()

## Summary

Key takeaways:
1. TF-IDF provides a simple yet effective representation for text classification.
2. All three classifiers achieve strong performance on well-separated categories.
3. SVM and Logistic Regression typically outperform Naive Bayes on TF-IDF features.
4. Per-class accuracy reveals which categories are harder to distinguish.